**Series 4 · ML Fundamentals** ‣ [04-1 ความรู้เบื้องต้นเกี่ยวกับ ML](./04-1-ml-introduction.ipynb) ・ [04-2 Train/Test Split & Overfitting](./04-2-train-test-split-and-overfitting.ipynb) ・ [04-3 Pipeline & Data Leakage](./04-3-pipeline-and-data-leakage.ipynb)

---

# 04-1: ความรู้เบื้องต้นเกี่ยวกับ Machine Learning
### และรูปแบบการเขียนโค้ดมาตรฐานของ scikit-learn

### วัตถุประสงค์การเรียนรู้
1. เข้าใจว่า Machine Learning ต่างจากการเขียนโปรแกรมแบบเดิมอย่างไร
2. แยกประเภทปัญหาได้: **Regression / Classification / Clustering**
3. แยก **Feature (X)** และ **Target (y)** ออกจากตารางข้อมูลได้
4. ใช้รูปแบบ `สร้างโมเดล → fit → predict` ของ scikit-learn ได้ทั้ง 3 ประเภท

## 1. ความหมายของ Machine Learning

| | การเขียนโปรแกรมแบบเดิม | Machine Learning |
|---|---|---|
| **ข้อมูลนำเข้า** | กฎ (Rules) + ข้อมูล | ข้อมูล + คำตอบในอดีต |
| **ผลลัพธ์ที่ได้** | คำตอบ | **กฎ (โมเดล)** ที่เรียนรู้เอง |
| **ตัวอย่าง** | `if อากาศ == 'rainy': คนน้อย` | ให้โมเดลเรียนรู้จากข้อมูล 365 วันและหาความสัมพันธ์เอง |

> **โมเดล (Model)** = สูตรคณิตศาสตร์ที่เรียนรู้ความสัมพันธ์ระหว่าง **ข้อมูลนำเข้า (X)** กับ **คำตอบ (y)** จากตัวอย่างในอดีต

## 2. ประเภทของ Machine Learning

| ประเภท | มีคำตอบ (y) หรือไม่ | ชนิดของคำตอบ | ตัวอย่างคำถาม | โมเดลตัวอย่าง |
|---|---|---|---|---|
| **Supervised – Regression** | มี | ตัวเลขต่อเนื่อง | วันนี้จะมีผู้เล่นกี่คน? | Linear Regression, Random Forest Regressor |
| **Supervised – Classification** | มี | หมวดหมู่ | วันนี้มีผู้เล่นจำนวนมากหรือไม่ (Yes/No) | Logistic Regression, Decision Tree, KNN |
| **Unsupervised – Clustering** | ไม่มี | — (หากลุ่มเอง) | วันที่มีลักษณะคล้ายกันแบ่งได้เป็นกี่กลุ่ม | K-Means, DBSCAN |
| **Unsupervised – ลดมิติ** | ไม่มี | — | สามารถลด 10 คอลัมน์เหลือ 2 คอลัมน์ได้หรือไม่ | PCA |

### แนวทางการเลือกประเภทของปัญหาจากโจทย์
```
โจทย์มีคอลัมน์คำตอบ (Target) หรือไม่
├── ไม่มี  → Unsupervised (Clustering / PCA)
└── มี    → Target เป็นอะไร?
            ├── ตัวเลขต่อเนื่อง (ราคา, จำนวน, อุณหภูมิ)  → Regression
            └── หมวดหมู่ (Yes/No, ชนิด, เกรด)           → Classification
```
> **ข้อควรระวัง:** Target ที่เป็นตัวเลขแต่มีค่าไม่กี่ค่า เช่น `0/1` หรือ `1–5 ดาว` มักเป็น **Classification**

## 3. โหลดข้อมูล: Golf Play Dataset (1 ปี, 365 วัน)

เป็นข้อมูลชุดเดียวกับ Series 3 (Data Exploration) แต่ครั้งนี้เราจะนำมา **สร้างโมเดลทำนาย**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 📌 ถ้าใช้ Google Colab: อัปโหลดไฟล์ CSV ที่แผง Files (ซ้ายมือ)
#    แล้วเปลี่ยน DATA_PATH เป็น "golf_dataset_weather_tropical_1year.csv"
DATA_PATH = "Data/golf_dataset_weather_tropical_1year.csv"
df = pd.read_csv(DATA_PATH)
print("ขนาดข้อมูล:", df.shape)
df.head()

ขนาดข้อมูล: (365, 15)


,Date,SunshineDuration,Outlook,Temperature,Temperature_binned,Humidity,Humidity_binned,WindDirection,WindSpeed,WindSpeed_binned,Pressure,TotalPlayer,TotalPlayer_binned,TimePerRound,TimePerRound_binned
0,2021-01-01,1.7,rainy,27.2,Warm,79,Humid,62.0,1.3,Very Light Air,1010.3,28,Very Few,206,Fast
1,2021-01-02,11.8,sunny,33.4,Hot,70,Moderate,350.0,0.1,No Wind,1013.2,74,Many,453,Slow
2,2021-01-03,13.1,sunny,31.3,Hot,75,Humid,345.3,2.2,Light Air,1005.5,63,Moderate,290,Moderate
3,2021-01-04,0.8,overcast,26.8,Warm,68,Moderate,304.2,2.3,Light Air,1009.6,56,Moderate,298,Moderate
4,2021-01-05,5.1,sunny,30.8,Hot,59,Dry,0.1,6.2,Mild Breeze,1011.3,70,Many,213,Fast


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Date                 365 non-null    object 
 1   SunshineDuration     365 non-null    float64
 2   Outlook              365 non-null    object 
 3   Temperature          365 non-null    float64
 4   Temperature_binned   365 non-null    object 
 5   Humidity             365 non-null    int64  
 6   Humidity_binned      365 non-null    object 
 7   WindDirection        365 non-null    float64
 8   WindSpeed            365 non-null    float64
 9   WindSpeed_binned     361 non-null    object 
 10  Pressure             365 non-null    float64
 11  TotalPlayer          365 non-null    int64  
 12  TotalPlayer_binned   365 non-null    object 
 13  TimePerRound         365 non-null    int64  
 14  TimePerRound_binned  365 non-null    object 
dtypes: float64(5), int64(3), object(7)
memor

## 4. แยก Feature (X) และ Target (y)

- **Target (y)** = สิ่งที่อยากทำนาย → `TotalPlayer` (จำนวนผู้เล่นในวันนั้น)
- **Feature (X)** = ข้อมูลที่ใช้ทำนาย → ข้อมูลสภาพอากาศ

### คอลัมน์ที่ไม่ควรนำมาใช้เป็น Feature (X)
| คอลัมน์ | เหตุผลที่ยังไม่ใช้ |
|---|---|
| `Date` | เป็นตัวระบุแถว ไม่ใช่สาเหตุ (ถ้าจะใช้ต้องแปลงเป็นเดือน/วันในสัปดาห์ก่อน) |
| `TotalPlayer_binned` | **สร้างมาจาก Target เอง** → ก่อให้เกิด Data Leakage (ดูบท 04-3) |
| `TimePerRound` | รู้ค่าได้ **หลังจาก** เล่นจบวันแล้ว → ตอนทำนายจริงจะยังไม่มีข้อมูลนี้ |
| `Outlook` | เป็นข้อความ ต้อง Encode ก่อน (จะใส่ในบท 04-3 ด้วย Pipeline) |

> **หลักการสำคัญ:** ควรตั้งคำถามเสมอว่า *"ตอนที่ต้องทำนายจริง เราจะรู้ค่าคอลัมน์นี้หรือยัง?"*

In [3]:
feature_cols = ['SunshineDuration', 'Temperature', 'Humidity',
                'WindDirection', 'WindSpeed', 'Pressure']

X = df[feature_cols]     # ตัวพิมพ์ใหญ่ = ตาราง (2 มิติ)
y = df['TotalPlayer']    # ตัวพิมพ์เล็ก = คอลัมน์เดียว (1 มิติ)

print("X:", X.shape, "  y:", y.shape)
X.head()

X: (365, 6)   y: (365,)


,SunshineDuration,Temperature,Humidity,WindDirection,WindSpeed,Pressure
0,1.7,27.2,79,62.0,1.3,1010.3
1,11.8,33.4,70,350.0,0.1,1013.2
2,13.1,31.3,75,345.3,2.2,1005.5
3,0.8,26.8,68,304.2,2.3,1009.6
4,5.1,30.8,59,0.1,6.2,1011.3


## 5. รูปแบบมาตรฐานของ scikit-learn (ใช้ได้กับทุกโมเดล)

```python
from sklearn.xxx import ชื่อโมเดล

model = ชื่อโมเดล()          # 1) สร้างโมเดล (กำหนด hyperparameter ได้ในวงเล็บ)
model.fit(X_train, y_train)  # 2) ให้โมเดลเรียนรู้จากข้อมูล
y_pred = model.predict(X_test)  # 3) ทำนาย
```

### 5.1 Regression: ทำนายจำนวนผู้เล่น

In [4]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()   # 1) สร้างโมเดล
model.fit(X, y)              # 2) เรียนรู้
y_pred = model.predict(X)    # 3) ทำนาย

pd.DataFrame({'Actual': y.values[:8], 'Predicted': y_pred[:8].round(1)})

,Actual,Predicted
0,28,48.4
1,74,78.5
2,63,74.3
3,56,56.9
4,70,79.1
5,63,78.4
6,51,57.1
7,69,71.2


### สิ่งที่โมเดลเรียนรู้: ค่าสัมประสิทธิ์ (Coefficient)
ค่าบวก = ยิ่งค่านี้สูง ผู้เล่นยิ่งมาก / ค่าลบ = ยิ่งสูง ผู้เล่นยิ่งน้อย

In [5]:
coef = pd.Series(model.coef_, index=feature_cols).round(2).sort_values()
print(coef)
print("\nค่าคงที่ (intercept):", round(model.intercept_, 2))

Humidity           -0.92
Pressure           -0.58
WindDirection      -0.00
WindSpeed           0.17
SunshineDuration    0.79
Temperature         2.61
dtype: float64

ค่าคงที่ (intercept): 629.44


### ทำนายวันใหม่ที่โมเดลไม่เคยเห็น
ข้อมูลใหม่ต้องมี **คอลัมน์ชื่อเดียวกันและลำดับเดียวกัน** กับตอน `fit`

In [6]:
new_days = pd.DataFrame({
    'SunshineDuration': [10.0, 1.0],
    'Temperature':      [30.0, 24.0],
    'Humidity':         [65,   90],
    'WindDirection':    [180.0, 180.0],
    'WindSpeed':        [2.0,  2.0],
    'Pressure':         [1009.0, 1009.0],
}, index=['วันแดดจ้า', 'วันครึ้มชื้น'])

new_days['Predicted_Players'] = model.predict(new_days[feature_cols]).round(0)
new_days

,SunshineDuration,Temperature,Humidity,WindDirection,WindSpeed,Pressure,Predicted_Players
วันแดดจ้า,10.0,30.0,65,180.0,2.0,1009.0,76.0
วันครึ้มชื้น,1.0,24.0,90,180.0,2.0,1009.0,30.0


### ข้อจำกัด: การวัดผลบนข้อมูลที่ใช้ฝึกโมเดล
`model.score()` ของ Regression คืนค่า **R²** (ยิ่งใกล้ 1 ยิ่งดี) แต่ในขั้นนี้เป็นการวัดบนข้อมูลชุดเดียวกับที่ใช้ฝึก
เปรียบเสมือน **การออกข้อสอบจากแบบฝึกหัดที่มีเฉลยแล้ว** → คะแนนที่ได้จึงเชื่อถือไม่ได้ → แก้ด้วย **Train/Test Split** ในบท 04-2

In [7]:
print("R² (วัดบนข้อมูลที่ใช้สอน — ยังเชื่อถือไม่ได้):", round(model.score(X, y), 3))

R² (วัดบนข้อมูลที่ใช้สอน — ยังเชื่อถือไม่ได้): 0.568


### 5.2 Classification: การทำนายว่าวันนี้มีผู้เล่นจำนวนมากหรือไม่
สร้าง Target ใหม่ `IsBusy` = 1 ถ้าผู้เล่น ≥ 60 คน, 0 ถ้าน้อยกว่า

In [8]:
df['IsBusy'] = (df['TotalPlayer'] >= 60).astype(int)
y_class = df['IsBusy']
print(y_class.value_counts())
print("\nสัดส่วน:", y_class.value_counts(normalize=True).round(2).to_dict())

IsBusy
1    204
0    161
Name: count, dtype: int64

สัดส่วน: {1: 0.56, 0: 0.44}


In [9]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)   # 1) สร้าง (โครงเดิม แค่เปลี่ยนชื่อโมเดล)
clf.fit(X, y_class)                       # 2) เรียนรู้
print("ทำนายคลาส     :", clf.predict(new_days[feature_cols]))
print("ความน่าจะเป็น (คลาส 0, คลาส 1):")
print(clf.predict_proba(new_days[feature_cols]).round(3))

ทำนายคลาส     : [1 0]
ความน่าจะเป็น (คลาส 0, คลาส 1):
[[0.06  0.94 ]
 [0.928 0.072]]


> **หมายเหตุ:** Classification มีคำสั่งพิเศษ `predict_proba()` ที่บอก **ความมั่นใจ** ของโมเดลในแต่ละคลาส
> ซึ่งจะใช้ต่อในการวาด ROC Curve (ดู PreTest 01)

### 5.3 Clustering: การจัดกลุ่มโดยไม่มีคำตอบ
K-Means คำนวณ **ระยะทาง** จึงต้อง Scale ข้อมูลก่อน (ทบทวนบท 02-5)

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)  # 1) สร้าง
kmeans.fit(X_scaled)                                       # 2) เรียนรู้ (ไม่มี y!)
df['Cluster'] = kmeans.labels_                             # 3) ผลการจัดกลุ่ม

df.groupby('Cluster')[['SunshineDuration', 'Temperature', 'Humidity', 'TotalPlayer']].mean().round(1)

,SunshineDuration,Temperature,Humidity,TotalPlayer
Cluster,,,,
0,8.1,30.1,68.4,72.0
1,2.0,25.0,82.1,41.5
2,2.5,25.7,68.6,54.5


> **การตีความ:** พิจารณาค่าเฉลี่ยของแต่ละกลุ่มแล้วตั้งชื่อกลุ่ม เช่น *"กลุ่มแดดดี-ร้อน"* หรือ *"กลุ่มชื้น-ครึ้ม"*
> แม้ไม่ได้ใช้ `TotalPlayer` ในการจัดกลุ่ม แต่กลุ่มที่ได้มีจำนวนผู้เล่นต่างกันชัดเจน

---

## 6. สรุปคำสั่งสำคัญ

| คำสั่ง | ทำอะไร | ใช้กับ |
|---|---|---|
| `model.fit(X, y)` | ให้โมเดลเรียนรู้ | Supervised |
| `model.fit(X)` | เรียนรู้โดยไม่มีคำตอบ | Unsupervised |
| `model.predict(X)` | ทำนายคำตอบ | ทุกประเภท |
| `model.predict_proba(X)` | ความน่าจะเป็นของแต่ละคลาส | Classification |
| `model.score(X, y)` | คะแนนเบื้องต้น (R² หรือ Accuracy) | Supervised |
| `scaler.fit_transform(X)` | เรียนรู้ + แปลงข้อมูล | Preprocessing |
| `scaler.transform(X)` | แปลงด้วยค่าที่เรียนไว้แล้ว | Preprocessing |

## แบบฝึกหัด
1. เปลี่ยน `LinearRegression` เป็น `DecisionTreeRegressor` (จาก `sklearn.tree`) แล้วดู `score` บนข้อมูลชุดเดิม — ได้คะแนนเท่าใด และเหตุใดจึงสูงผิดปกติ
2. ทดลองเปลี่ยน `n_clusters` ของ K-Means เป็น 2 และ 4 แล้วเปรียบเทียบค่าเฉลี่ยแต่ละกลุ่ม

In [11]:
# เฉลยข้อ 1
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X, y)
print("R² ของ Decision Tree บนข้อมูลที่ใช้สอน:", tree.score(X, y))
# ได้ 1.0 (สมบูรณ์แบบ) เพราะต้นไม้ไม่จำกัดความลึกจะ 'จำ' ข้อมูลทุกแถว
# → นี่คือ Overfitting ซึ่งจะพิสูจน์ในบท 04-2 ว่าเมื่อเจอข้อมูลใหม่จะทำได้แย่ลง

R² ของ Decision Tree บนข้อมูลที่ใช้สอน: 1.0


---
**บทถัดไป:** [04-2 Train/Test Split & Overfitting](./04-2-train-test-split-and-overfitting.ipynb)